# Deploy MLflow on K8S

Create a folder for MLflow:
```bash
mkdir -p ~/k8s/mlflow
```

Create namespace:
```bash
sudo kubectl create namespace platform
```

Create MLflow deployment and service:
```bash
cat << 'EOF' > ~/k8s/mlflow/mlflow.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mlflow
  namespace: platform
spec:
  replicas: 1
  selector:
    matchLabels:
      app: mlflow
  template:
    metadata:
      labels:
        app: mlflow
    spec:
      containers:
      - name: mlflow
        image: ghcr.io/mlflow/mlflow:v2.19.0
        command: ["mlflow", "server", "--host", "0.0.0.0", "--port", "5000"]
        ports:
        - containerPort: 5000
        resources:
          requests:
            cpu: "50m"
            memory: "512Mi"
          limits:
            cpu: "500m"
            memory: "1Gi"
        volumeMounts:
        - name: mlflow-data
          mountPath: /mlflow
      volumes:
      - name: mlflow-data
        persistentVolumeClaim:
          claimName: mlflow-pvc
---
apiVersion: v1
kind: Service
metadata:
  name: mlflow
  namespace: platform
spec:
  type: NodePort
  selector:
    app: mlflow
  ports:
  - port: 5000
    targetPort: 5000
    nodePort: 30500
EOF
```

```bash
cat << 'EOF' > ~/k8s/mlflow/pv.yaml
apiVersion: v1
kind: PersistentVolume
metadata:
  name: mlflow-pv
spec:
  capacity:
    storage: 10Gi
  accessModes:
  - ReadWriteOnce
  hostPath:
    path: /mnt/mlflow
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: mlflow-pvc
  namespace: platform
spec:
  accessModes:
  - ReadWriteOnce
  resources:
    requests:
      storage: 10Gi
EOF
```

Deploy MLflow:
```bash
sudo kubectl apply -f ~/k8s/mlflow/pv.yaml
sudo kubectl apply -f ~/k8s/mlflow/mlflow.yaml
```

Verify deployment:
```bash
sudo kubectl get pods -n platform
```

Verify persistence (restart pod and confirm MLflow is still accessible):
```bash
sudo kubectl rollout restart deployment mlflow -n platform
sudo kubectl get pods -n platform
```

Substitute `A.B.C.D` with `floating ip`. Access MLflow UI at

http://A.B.C.D:30500